# Links 
github repo: https://github.com/vildeskobba/IND320vilde 

Streamlit app: https://ind320vilde.streamlit.app/ 

Panopto_video: 


In [ ]:
%pip install -q -r requirements.txt

In [1]:
#importing libraris
import pandas as pd
import requests

In [23]:
def get_exchange_rates(url:str):
    response = requests.get(
        url,
        params = {
            "format": "sdmx-json",
            "locale": "no",
        },
        timeout = 30,
    )
    response.raise_for_status() #raise error if request was unsuccessfull 
    data = response.json()["data"] # convert json to dict

    dimensions = data["structure"]["dimensions"]
    series_dims = dimensions["series"]

    currencies = {
        d["id"]: d["values"] for d in series_dims
    }

    dates = dimensions["observation"][0]["values"]

    records = []
    for key, series in data["dataSets"][0]["series"].items():
        indices = key.split(":")
        base = currencies["BASE_CUR"][int(indices[2])]["id"]
        quote = currencies["QUOTE_CUR"][int(indices[3])]["id"]

        for date, values in series["observations"].items():
            records.append([
                dates[int(date)]["id"], base, quote, values[0]
            ])

    df = pd.DataFrame(
        records,
        columns=["TIME_PERIOD", "BASE_CUR", "QUOTE_CUR", "OBS_VALUE"]
        )

    df["TIME_PERIOD"] = pd.to_datetime(df["TIME_PERIOD"]) 

    return df
    

In [24]:
exchange_rates_df = get_exchange_rates("https://data.norges-bank.no/api/data/EXR/B.USD.NOK.SP?endPeriod=2026-11-17&format=sdmx-json&locale=no&startPeriod=2024-01-01")

print(exchange_rates_df.head())

  TIME_PERIOD BASE_CUR QUOTE_CUR OBS_VALUE
0  2024-01-02      USD       NOK   10.2971
1  2024-01-03      USD       NOK   10.3672
2  2024-01-04      USD       NOK   10.3027
3  2024-01-05      USD       NOK   10.3553
4  2024-01-08      USD       NOK     10.38


In [25]:
exchange_rates_df = get_exchange_rates("https://data.norges-bank.no/api/data/EXR/B.EUR.NOK.SP?endPeriod=2026-11-17&format=sdmx-json&locale=no&startPeriod=2024-01-01")

print(exchange_rates_df.head())

  TIME_PERIOD BASE_CUR QUOTE_CUR OBS_VALUE
0  2024-01-02      EUR       NOK   11.2815
1  2024-01-03      EUR       NOK     11.32
2  2024-01-04      EUR       NOK   11.2845
3  2024-01-05      EUR       NOK    11.309
4  2024-01-08      EUR       NOK    11.362
